# STAT163 · Week 4 · Before the lecture: working with text in columns

Plan about **45 minutes** for this notebook, and work through it before the lecture.

You will:

- write a rule for the shape of a text value, and count the values that have that shape
- turn such a rule into a column
- find the rows that match no rule, which is where wrong numbers come from
- decide what a missing value means, and see what you lose with each way of handling it

**How to use it.** Run the cells one at a time, from top to bottom. Three kinds of cells:

- **Read and run.** Run the cell and read the output. Most cells are this kind.
- **Predict.** Answer the question in the `# Predict:` comment before you run the cell.
  Write your answer under it, on a new line that starts with `#`. The answer is under
  **Answer** below the cell. Open it after you run.
- **Try it.** Follow the instruction in the cell's comment: change the code, then run it.

Nothing here is submitted or graded.

AI is welcome here. Ask it to explain a cell, or why your prediction was wrong.

## Load the table

The page of the dataset in the UCI Machine Learning Repository,
[Online Retail II](https://archive.ics.uci.edu/dataset/502/online+retail+ii), describes the
columns like this:

> - **InvoiceNo**: Invoice number. Nominal. A 6-digit integral number uniquely assigned to
>   each transaction. If this code starts with the letter 'c', it indicates a cancellation.
> - **StockCode**: Product (item) code. Nominal. A 5-digit integral number uniquely
>   assigned to each distinct product.
> - **Description**: Product (item) name. Nominal.
> - **Quantity**: The quantities of each product (item) per transaction. Numeric.
> - **InvoiceDate**: Invice date and time. Numeric. The day and time when a transaction was
>   generated.
> - **UnitPrice**: Unit price. Numeric. Product price per unit in sterling (£).
> - **CustomerID**: Customer number. Nominal. A 5-digit integral number uniquely assigned
>   to each customer.
> - **Country**: Country name. Nominal. The name of the country where a customer resides.

In the file, three of the names are written differently: `Invoice`, `Price` and
`Customer ID`.

In [1]:
import numpy as np
import pandas as pd

url = "https://raw.githubusercontent.com/stat163-2026t1/week3-pre-lecture/main/data/online_retail_2010_11.csv"
df = pd.read_csv(url)

df["Customer ID"] = df["Customer ID"].astype("Int64")
df["line_revenue"] = df["Quantity"] * df["Price"]
df.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,line_revenue
0,529995,48184,DOORMAT ENGLISH ROSE,6,2010-11-01 08:56:00,7.95,16316,United Kingdom,47.7
1,529995,48187,DOORMAT NEW ENGLAND,4,2010-11-01 08:56:00,7.95,16316,United Kingdom,31.8
2,529995,21523,DOORMAT FANCY FONT HOME SWEET HOME,10,2010-11-01 08:56:00,6.75,16316,United Kingdom,67.5
3,529995,22708,WRAP DOLLY GIRL,25,2010-11-01 08:56:00,0.42,16316,United Kingdom,10.5
4,529995,22781,GUMBALL MAGAZINE RACK,4,2010-11-01 08:56:00,7.65,16316,United Kingdom,30.6


## Part 1 — Checking a text column against its description

The documentation says that `StockCode` is "a 5-digit integral number uniquely assigned to
each distinct product". Does every code in the table have that shape? If not, how many
codes do not, and what are they?

To check the shape of a whole value, we write a **pattern**. With `fullmatch` you check
whether a value has that shape from its first character to its last. In a pattern, `\d`
stands for one digit, and with `{5}` you repeat the piece before it five times, so `\d{5}`
means five digits:

In [2]:
code = df["StockCode"]
is_five_digits = code.str.fullmatch(r"\d{5}")

print(len(code))
print(is_five_digits.sum())

78015
69026


We write the pattern inside `r"..."`, a **raw string**. In a raw string a backslash is an
ordinary character, so `\d` stays in the pattern as we wrote it.

These are the pieces of a pattern we use in this notebook:

| Piece | Means |
|---|---|
| `\d` | one digit |
| `{5}` | the piece before it, five times |
| `[A-Z]` | one capital letter |
| `[A-Za-z]` | one letter, capital or small |
| `?` | the piece before it, zero times or once |

Another name for a pattern is a **regular expression**, or **regex** for short. The text you
pass to `contains` and `fullmatch` is read as a pattern. To practise reading patterns,
[RegexLearn](https://regexlearn.com/learn/regex101) has a short interactive course for
beginners, and on [regex101](https://regex101.com) you can paste a pattern and some text
and see which parts match.

Most codes fit the description, but not all of them. Which codes do not? Here are the most
common ones:

In [3]:
code[~is_five_digits].value_counts().head(10)

StockCode
85123A    423
85099B    270
84029E    183
84029G    180
85099C    120
POST      120
82494L    114
85049E    106
84970S    105
47591D    102
Name: count, dtype: int64

Most of them are five digits and a capital letter. What does the letter mean? Read the
names of the codes that start with `85099`:

In [4]:
df.loc[code.str.fullmatch(r"85099[A-Z]"), ["StockCode", "Description"]].drop_duplicates()

,StockCode,Description
344,85099F,JUMBO BAG STRAWBERRY
346,85099B,JUMBO BAG RED RETROSPOT
886,85099C,JUMBO BAG BAROQUE BLACK WHITE


The letter marks a variant of one product: the same jumbo bag in three designs. That is the
first finding. Now we take these codes out as well and look at what is left.

In [ ]:
# Predict: what kinds of code are left once we also take out five digits with a capital
# letter?
is_variant = code.str.fullmatch(r"\d{5}[A-Z]")
left = code[~is_five_digits & ~is_variant]

print(len(left))
print(sorted(left.unique()))

<details>
<summary>Answer</summary>

Three kinds:

- the same shape with a small letter, such as `72349b`, and `85123a`, which is the same
  product as `85123A`
- two letters after the digits, such as `15056BL`
- codes that are not products at all: `POST`, `M`, `DOT`, `BANK CHARGES`, `AMAZONFEE`, and
  the gift vouchers

When you write a pattern, you make a claim about every value in the column. A small letter
is enough to break it, and you get no warning.

</details>

With `[A-Za-z]` we allow a small letter too. Take those codes out and see how much is left:

In [6]:
is_variant = code.str.fullmatch(r"\d{5}[A-Za-z]")
left = code[~is_five_digits & ~is_variant]

print(len(left))
left.value_counts()

420


StockCode
POST            120
15056BL          85
M                77
DOT              59
C2               30
D                20
15056bl           6
DCGSSBOY          4
DCGSSGIRL         4
DCGS0003          3
BANK CHARGES      3
AMAZONFEE         2
PADS              2
SP1002            1
gift_0001_30      1
gift_0001_10      1
DCGS0076          1
gift_0001_20      1
Name: count, dtype: int64

What is left is short enough to read in full: two-letter codes such as `15056BL`, and codes
that are not products, such as `POST` and `M`. With each step we took out one shape and
found the next one. With a text column you often work like this: take out what you
understand, then read the rest.

Count the two-letter codes yourself. In the next cell we count the codes with one letter
after the digits:

In [7]:
# Try it: change the pattern to count the codes with two letters after the digits
code.str.fullmatch(r"\d{5}[A-Za-z]").sum()

np.int64(8569)

### Full match or partial?

With `fullmatch` you check the whole value. With `contains` you check whether the pattern
occurs anywhere inside it.

In [8]:
print(code.str.contains(r"\d{5}").sum())
print(code.str.fullmatch(r"\d{5}[A-Za-z]?").sum())

77686
77595


Each count is the answer to a different question: five digits anywhere in the code, or a
code that is five digits with at most one letter after them. Decide which question you are
asking before you choose the method.

### Counting products by a word in their name

The same choice comes up with product names. How many products are pens?

In [ ]:
# Predict: besides pens, which products have the letters PEN in their name?
names = df["Description"].dropna().drop_duplicates()
is_pen = names.str.contains("PEN")

print(is_pen.sum())
names[is_pen].sort_values().head(20)

<details>
<summary>Answer</summary>

Pencils, pencil sharpeners and a pendant on a necklace are already in the first twenty.
With `contains` you also find the letters when they sit inside a longer word, and from the
count alone you cannot see that.

</details>

Take out the two biggest groups, pencils and pendants, and read what is left:

In [10]:
is_pen = (
    names.str.contains("PEN")
    & ~names.str.contains("PENCIL")
    & ~names.str.contains("PENDANT")
)
names[is_pen].sort_values()

325                  10 COLOUR SPACEBOY PEN
5249              ASSORTED TUTTI FRUTTI PEN
615             ASSTD DESIGN RACING CAR PEN
31999        BLUE  DIAMANTE PEN IN GIFT BOX
5017                 FEATHER PEN,COAL BLACK
3223                   FEATHER PEN,HOT PINK
5015                 FEATHER PEN,LIGHT PINK
2587                         FUNKY DIVA PEN
7031        GREEN  DIAMANTE PEN IN GIFT BOX
2930             HAND OPEN SHAPE DECO.WHITE
10390                  HAND OPEN SHAPE GOLD
7029         LILAC DIAMANTE PEN IN GIFT BOX
2121                 LIPSTICK PEN BABY PINK
2122                   LIPSTICK PEN FUSCHIA
553                        LIPSTICK PEN RED
8866                  MINI HIGHLIGHTER PENS
69526                OPEN CLOSED METAL SIGN
5388           PENNY FARTHING BIRTHDAY CARD
34085          PENS ASSORTED FUNKY JEWELED 
698                PENS ASSORTED FUNNY FACE
24975               PENS ASSORTED SPACEBALL
4603          PINK DIAMANTE PEN IN GIFT BOX
55415                         PO

A few names are still not pens: the two `HAND OPEN SHAPE` decorations, the
`OPEN CLOSED METAL SIGN`, a `PENNY FARTHING` birthday card, a `PENNANT` and a tape
`DISPENSER`. You could take them out too, in one more step. Stop
when the errors that are left are ones you can accept for your question, and say what they
are when you report the count.

## Part 2 — From a rule to a column

### Conditionally extracting the value into a new column

You have used `where` before: with it you keep a value in the rows where a condition is
True, and the other rows become missing. Here we keep the quantity only in the rows whose
code is five digits with at most one letter:

In [11]:
is_product = code.str.fullmatch(r"\d{5}[A-Za-z]?")
product_qty = df["Quantity"].where(is_product)

print(product_qty.dtype)
print(product_qty.isna().sum())

float64
420


`Quantity` has whole numbers, and the result is `float64`, the type for decimals. In pandas,
a whole-number column with missing values becomes a column of decimals. To get the whole
numbers back, convert it to `Int64`:

In [12]:
product_qty.astype("Int64").head(3)

0     6
1     4
2    10
Name: Quantity, dtype: Int64

The missing values are in the rows that do not match the rule.

There is an alternative way to achieve the same result: `np.where(condition, a, b)`. With it
you get `a` where the condition is True and `b` elsewhere, and the result is a NumPy array,
which has no row labels.

### Several results for several conditions

Often you have not one condition but several, and they have to be checked in order. In most
programming languages you would write
`if <condition> then <result> elseif <condition2> then <result2> ... else <default> end`.
If you know SQL, you would write
`CASE WHEN <condition> THEN <result> WHEN <condition2> THEN <result2> ... ELSE <default> END`.
In pandas, most of the time you use `np.select`.

For `np.select` you write a **list** of conditions and a **list** of labels. Each row gets
the label of the first condition that is True for it, and the rows where no condition is
True get the `default` label. The result is again an array, and we store it as a new
column. We write the default label in brackets, `(other)`, so that it comes first or last
when you sort the labels.

In [13]:
df["code_kind"] = np.select(
    [code.str.fullmatch(r"\d{5}"), code.str.fullmatch(r"\d{5}[A-Za-z]")],
    ["product", "product variant"],
    default="(other)",
)
df["code_kind"].value_counts()

code_kind
product            69026
product variant     8569
(other)              420
Name: count, dtype: int64

Now each row has its kind in one column: product, product variant or `(other)`. You could
also keep a True/False column for each kind, but a single column like this one is what you
need as the key of a groupby.

What happens when a row meets more than one condition?

In [ ]:
# Predict: how many rows get the label "exactly five digits"?
pd.Series(np.select(
    [code.str.contains(r"\d{5}"), code.str.fullmatch(r"\d{5}")],
    ["holds five digits", "exactly five digits"],
    default="(other)",
)).value_counts()

<details>
<summary>Answer</summary>

None. Every value that is exactly five digits also has five digits somewhere, so it meets
the first condition, and each row gets the label of the first condition it meets. No row
can get the second label, and you get no warning.

So put the narrowest condition first and the widest last.

</details>

### Writing multi-step transformations

So far you added a column with `df["new_col"] = <logic for the new column>`, as the load
cell does for `line_revenue`. With the `assign` method you add a column and get the whole
table back, so you can write several steps as one chain and use the new column in the
steps that follow. The round brackets around a chain let you write one step per line.

Step one: mark the product rows. We build the column from `code`, which exists before the
chain starts:

In [15]:
(
    df
    .assign(is_product=code.str.fullmatch(r"\d{5}[A-Za-z]?"))
    [["StockCode", "Quantity", "is_product"]]
    .sort_values("is_product")
    .head(3)
)

,StockCode,Quantity,is_product
58070,C2,1,False
15674,DOT,1,False
1748,C2,1,False


Step two: keep the quantity only on the product rows. This step uses `is_product`, which
exists only inside the chain, so we write it with a `lambda`, a small function written in
one line. In `lambda t: ...`, `t` is the table as it is at that point in the chain, and the
part after the colon is the new column:

In [16]:
(
    df
    .assign(is_product=code.str.fullmatch(r"\d{5}[A-Za-z]?"))
    .assign(product_qty=lambda t: t["Quantity"].where(t["is_product"]).astype("Int64"))
    [["StockCode", "Quantity", "is_product", "product_qty"]]
    .sort_values("is_product")
    .head(3)
)

,StockCode,Quantity,is_product,product_qty
58070,C2,1,False,<NA>
15674,DOT,1,False,<NA>
1748,C2,1,False,<NA>


Step three: add the quantity up by country.

In [17]:
(
    df
    .assign(is_product=code.str.fullmatch(r"\d{5}[A-Za-z]?"))
    .assign(product_qty=lambda t: t["Quantity"].where(t["is_product"]).astype("Int64"))
    .groupby("Country")["product_qty"]
    .sum()
    .sort_values(ascending=False)
    .head(3)
)

Country
United Kingdom    573350
Netherlands        20864
Germany            15380
Name: product_qty, dtype: Int64

This is the number of product units for each country, with cancelled units subtracted.
Without `where`, we would count postage and the other lines that are not products as units
too.

## Part 3 — The rows that match no rule

Whatever rule you write, some rows will not match it, and those are the rows to read first.
In Part 1 you read their codes. Now read the product name beside each code:

In [18]:
df.loc[df["code_kind"] == "(other)", ["StockCode", "Description"]].drop_duplicates()

,StockCode,Description
39,M,Manual
60,DOT,DOTCOM POSTAGE
692,DCGSSBOY,BOYS PARTY BAG
693,DCGSSGIRL,GIRLS PARTY BAG
1599,POST,POSTAGE
1653,C2,CARRIAGE
1771,SP1002,NaN
2733,D,Discount
2855,15056BL,EDWARDIAN PARASOL BLACK
5113,C2,NaN


These rows are of two different kinds.

- Lines that are not products: postage, shipping, a discount, a manual entry. These belong
  outside a count of products.
- Products whose code has another shape: the party bags `DCGSSBOY` and `DCGSSGIRL`, and
  `15056BL` and `15056bl`, with two letters in the code. For these, our rule is too narrow.

We can see both kinds because we gave these rows a label of their own, `(other)`.

### Missing values written as text

With `isna()` you find the cells that are empty. `Country` has no empty cells:

In [19]:
df["Country"].isna().sum()

np.int64(0)

Read the list of countries. Which value does not name a country?

In [20]:
sorted(df["Country"].unique())

['Australia',
 'Austria',
 'Belgium',
 'Canada',
 'Channel Islands',
 'Cyprus',
 'Denmark',
 'EIRE',
 'Finland',
 'France',
 'Germany',
 'Greece',
 'Israel',
 'Italy',
 'Japan',
 'Korea',
 'Lithuania',
 'Netherlands',
 'Norway',
 'Poland',
 'Portugal',
 'RSA',
 'Spain',
 'Sweden',
 'Switzerland',
 'USA',
 'United Arab Emirates',
 'United Kingdom',
 'Unspecified']

<details>
<summary>Answer</summary>

`Unspecified`. `EIRE` and `RSA` are countries too, Ireland and South Africa. `Unspecified`
says that the country is not known: it is a missing value that somebody wrote down as
text. If you count with `isna()` alone, you get zero missing countries.

</details>

Product names can hide the same thing. Read the shortest ones:

In [21]:
names[names.str.len() <= 8].sort_values()

2206            ?
1653     CARRIAGE
72510     Damaged
56693     Damages
2733     Discount
66774     Mailout
39         Manual
1599      POSTAGE
6514      damaged
3251      damages
66772     mailout
11975     rex use
Name: Description, dtype: str

`?` is a product name that nobody knew. Names such as `damaged`, `Discount` and `mailout`
are notes about a line, not products.

Turn `Unspecified` into a missing value on purpose:

In [22]:
df["Country"] = df["Country"].replace("Unspecified", pd.NA)
df["Country"].isna().sum()

np.int64(51)

With `pd.NA` you write a missing value into the column, and now these rows count as missing
for `isna()`. When you group on `Country`, they are left out, because rows with a missing
key get no group.

### What a missing value means

Some rows have no `Customer ID`. To understand what the missing value means, look at which
rows they are. We split the table into sales, cancellations and write-offs with
`np.select`, this time with conditions on the invoice number and the price:

In [23]:
df["row_kind"] = np.select(
    [df["Invoice"].str.startswith("C"), df["Price"] == 0],
    ["cancellation", "write-off"],
    default="sale",
)
df["row_kind"].value_counts()

row_kind
sale            76464
cancellation     1194
write-off         357
Name: count, dtype: int64

A write-off is a line with a price of zero.

A **flag** is a True/False column: True where the value is there, False where it is
missing.

In [24]:
df["has_customer"] = df["Customer ID"].notna()
df.groupby("row_kind")["has_customer"].mean().round(3)

row_kind
cancellation    0.983
sale            0.789
write-off       0.011
Name: has_customer, dtype: float64

A write-off has no buyer, so on those rows the missing value means "does not apply". On a
sale the missing value means "the shop did not record it", and that is one sale line in
five. So in one column a missing value has two different meanings, and you handle each one
differently.

There are three ways to handle the missing ids, and with each one you lose something:

In [25]:
kept = df.dropna(subset=["Customer ID"])
round(kept["line_revenue"].sum() / df["line_revenue"].sum(), 3)

np.float64(0.798)

**Drop** the rows without a customer, and you lose around 20% of the month's revenue. Every
number you compute on them is about known customers only, and you have to report it that
way.

In [ ]:
# Predict: how many customers does the table have before and after the missing ids are
# filled with 0?
filled = df["Customer ID"].fillna(0)
print(df["Customer ID"].nunique())
print(filled.nunique())

<details>
<summary>Answer</summary>

One more. When you **fill** the missing ids with 0, you turn every row without an id into
one customer, number 0.

</details>

That one customer is now the largest in the shop:

In [27]:
df.groupby(filled)["line_revenue"].sum().sort_values(ascending=False).head(3)

Customer ID
0        287775.36
14646     30810.51
15838     23165.86
Name: line_revenue, dtype: float64

Id 0 has nine times the revenue of the largest real customer, so every per-customer number
you compute after the fill is wrong.

**Flag**: keep every row, and keep the fact that the value is missing in a column of its
own, such as `has_customer` above:

In [28]:
df.groupby("has_customer")["line_revenue"].agg(["sum", "size"]).round(2)

,sum,size
has_customer,,
False,287775.36,16525
True,1134879.28,61490


With the flag you can see how much of the month's revenue comes from rows with a customer
and how much from rows without one, and every customer id stays as it was. After the drop
those rows are gone, and after the fill they look like one real customer.

Which of the three to use depends on your question. Drop the rows when your question is
only about rows that have the value. Fill only when you know the true value. Flag when you
want to count or compare the missing values themselves.

## What to check before you trust a text column

- What does the documentation say the values look like, and how many values fit?
- Whole value or anywhere inside it? A word, or letters inside a longer word?
- How many rows do not match my rule, and what is in them?
- Does any value mean "missing" without being missing?
- If I dropped or filled missing values, what is the number now about?

In this notebook you built three columns:

- the kind of each row, from rules about the text
- the quantity, kept only where a rule holds
- a flag for a missing customer id

**Bring to the lecture:** one column from this notebook where you would write the rule
differently, and the count that made you think so. The kind of product code is one
example: some of the rows labelled `(other)` are products.

Before the practice, work through the second notebook in this repository,
`02-before-the-practice-regex-and-split.ipynb`.